# Archaeology V6 — corrected + attention-pooling CNN

A clean rebuild that folds in everything the EDA established. Runs on Kaggle against
the `2nd-variation` (`FINALCNNDATA_NEW_01`) dataset.

**What this fixes / adds vs the old notebooks**
1. **Correct channel order** — `[Blue,Green,Red,NIR,LRM,Slope,SVF,HS1-4]`. The old
   fusion notebooks indexed the wrong bands (LRM/SVF/Slope/NDVI were computed on the wrong
   channels). Fixed here and used consistently.
2. **Real nodata handling** — `-9999`/NaN/Inf are masked to NaN, filled with the
   per-channel *training* mean, and an explicit valid-mask is available. (Old code did
   `nan_to_num(-9999 -> 0)`, poisoning normalization.)
3. **Attention / MIL pooling head** — instead of global average pooling. The EDA showed
   53% of "site" patches contain a *single* monument, so the positive signal is spatially
   tiny; attention lets the model localize it instead of averaging it away.
4. **Monument-count-aware training** — positives are sample-weighted by monument_count
   (a 42-monument patch is a far stronger label than a 1-monument patch).
5. **Group-held-out evaluation** — 5-fold `StratifiedGroupKFold` on `group_id`, frozen
   test set, ensemble + OOF threshold. No spatial leakage.
6. **Locked reference lines** — LogReg 0.657, best plain CNN ~0.68. Anything below ~0.68
   on group-CV is not progress.

Run top to bottom.


In [1]:
# ============================================================
# CELL 1 — CONFIG + DEVICE + SEED
# ============================================================
import os, glob, random, math, warnings
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
warnings.filterwarnings("ignore")

CONFIG = {
    "n_channels": 11,
    "patch_size": 250,
    "nodata_value": -9999.0,

    # correct 0-indexed channel order (confirmed by EDA cell 4)
    "channel_names": ["Blue","Green","Red","NIR","LRM","Slope","SVF",
                      "HS1","HS2","HS3","HS4"],

    # model
    "base_width": 32,          # first conv block width
    "dropout": 0.35,
    "pool": "attention",       # "attention" (gated-MIL) or "gap"

    # training
    "batch_size": 16,
    "epochs": 40,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "early_stop_patience": 8,

    # imbalance / labels
    "use_weighted_sampler": True,
    "count_weight_cap": 8.0,   # cap monument_count influence on sample weight
    "count_weight_alpha": 0.5, # weight ~ 1 + alpha*log1p(count), capped

    # CV
    "n_folds": 5,
    "seed": 42,

    "ckpt_dir": "/kaggle/working/v6_ckpts",
    "threshold_grid": np.round(np.arange(0.10, 0.71, 0.01), 2),
}
os.makedirs(CONFIG["ckpt_dir"], exist_ok=True)

CH = {n.upper(): i for i, n in enumerate(CONFIG["channel_names"])}
CH["HS1"], CH["HS2"], CH["HS3"], CH["HS4"] = 7, 8, 9, 10  # explicit

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(CONFIG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.10.0+cu128 | Device: cuda
GPU: Tesla T4


## 2 — Find dataset, load metadata (labels, counts, groups)

In [2]:
# ============================================================
# CELL 2 — DATASET DISCOVERY + METADATA
# ============================================================
train_dirs = []
for root, dirs, files in os.walk("/kaggle/input"):
    for d in dirs:
        if d.lower() == "train":
            train_dirs.append(os.path.join(root, d))
assert train_dirs, "No 'train' dir under /kaggle/input — attach the dataset."
DATA_ROOT = os.path.dirname(train_dirs[0])
print("DATA_ROOT:", DATA_ROOT)

meta_path = glob.glob(os.path.join(DATA_ROOT, "**", "metadata_all.csv"), recursive=True)[0]
meta = pd.read_csv(meta_path)
meta["_base"] = meta["patch_name"].astype(str).apply(os.path.basename)
for c in ["label", "monument_count"]:
    meta[c] = pd.to_numeric(meta[c], errors="coerce").fillna(0).astype(int)
GROUP_COL = "group_id"

meta_lookup = {r["_base"]: {"label": int(r["label"]),
                            "count": int(r["monument_count"]),
                            "group": r[GROUP_COL]}
               for _, r in meta.iterrows()}
print("metadata rows:", len(meta), "| groups:", meta[GROUP_COL].nunique())

def collect_split(split):
    recs = []
    for cls, lab in [("site", 1), ("no_site", 0)]:
        d = os.path.join(DATA_ROOT, split, cls)
        for f in sorted(glob.glob(os.path.join(d, "*.tif")) +
                        glob.glob(os.path.join(d, "*.tiff"))):
            b = os.path.basename(f)
            m = meta_lookup.get(b, {"label": lab, "count": lab, "group": -1})
            recs.append({"path": f, "label": lab, "count": m["count"], "group": m["group"]})
    return recs

train_recs = collect_split("train")
val_recs   = collect_split("validation")
test_recs  = collect_split("test")
trainval   = train_recs + val_recs

missing_grp = sum(1 for r in trainval if r["group"] in (-1, "-1"))
print(f"train+val: {len(trainval)} | test: {len(test_recs)} | "
      f"trainval without group: {missing_grp}")
assert missing_grp == 0, "Some train+val patches have no group_id — grouping would leak."


DATA_ROOT: /kaggle/input/datasets/shreyansdeshpande/2nd-variation/FINALCNNDATA_NEW_01
metadata rows: 3617 | groups: 40
train+val: 3117 | test: 500 | trainval without group: 0


## 3 — Nodata-aware loader + per-channel training stats

In [3]:
# ============================================================
# CELL 3 — LOADER (mask -9999) + CHANNEL STATS
# ============================================================
NODATA = CONFIG["nodata_value"]
C, P = CONFIG["n_channels"], CONFIG["patch_size"]

def load_raw(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)           # (C,H,W)
    invalid = (~np.isfinite(arr)) | (arr == NODATA)
    arr[invalid] = np.nan
    return arr, invalid

def compute_channel_stats(recs, max_samples=1000):
    idx = np.random.default_rng(CONFIG["seed"]).choice(
        len(recs), size=min(max_samples, len(recs)), replace=False)
    s = np.zeros(C); s2 = np.zeros(C); n = np.zeros(C)
    for i in idx:
        arr, _ = load_raw(recs[i]["path"])
        for c in range(C):
            v = arr[c][np.isfinite(arr[c])]
            s[c] += v.sum(); s2[c] += (v**2).sum(); n[c] += v.size
    mean = s / np.maximum(n, 1)
    std = np.sqrt(np.maximum(s2/np.maximum(n,1) - mean**2, 1e-8))
    return mean.astype(np.float32), std.astype(np.float32)

print("Computing per-channel training stats (nodata-masked)...")
CH_MEAN, CH_STD = compute_channel_stats(train_recs)
for i, nm in enumerate(CONFIG["channel_names"]):
    print(f"  {i:2d} {nm:6s} mean={CH_MEAN[i]:.4f} std={CH_STD[i]:.4f}")


Computing per-channel training stats (nodata-masked)...
   0 Blue   mean=0.0357 std=0.0081
   1 Green  mean=0.0444 std=0.0165
   2 Red    mean=0.0362 std=0.0138
   3 NIR    mean=0.2555 std=0.0995
   4 LRM    mean=0.0159 std=0.7493
   5 Slope  mean=7.2645 std=6.2608
   6 SVF    mean=0.9314 std=0.0521
   7 HS1    mean=0.6357 std=0.0879
   8 HS2    mean=0.6315 std=0.0909
   9 HS3    mean=0.6322 std=0.0879
  10 HS4    mean=0.6366 std=0.0906


## 4 — Dataset: fill nodata with train mean, z-score, augment, return valid-mask

In [4]:
# ============================================================
# CELL 4 — TERRAIN DATASET
# ============================================================
class TerrainDataset(Dataset):
    def __init__(self, records, mean, std, augment=False):
        self.records = records
        self.mean = mean.reshape(C, 1, 1)
        self.std  = std.reshape(C, 1, 1)
        self.augment = augment

    def __len__(self): return len(self.records)

    def _augment(self, x):
        if random.random() < 0.5: x = np.flip(x, 2).copy()
        if random.random() < 0.5: x = np.flip(x, 1).copy()
        k = random.randint(0, 3)
        if k: x = np.rot90(x, k, axes=(1, 2)).copy()
        return x

    def __getitem__(self, i):
        r = self.records[i]
        arr, invalid = load_raw(r["path"])
        # fill invalid with per-channel train mean (pre-normalization)
        mean_fill = np.broadcast_to(self.mean, arr.shape)
        arr = np.where(np.isfinite(arr), arr, mean_fill).astype(np.float32)
        # z-score with train stats
        arr = (arr - self.mean) / self.std
        valid = (~invalid.any(0)).astype(np.float32)     # (H,W) 1=valid
        if self.augment:
            stacked = np.concatenate([arr, valid[None]], 0)
            stacked = self._augment(stacked)
            arr, valid = stacked[:C], stacked[C]
        return (torch.from_numpy(arr).float(),
                torch.tensor(r["label"], dtype=torch.float32),
                torch.from_numpy(valid).float())

def make_sample_weights(records):
    labels = np.array([r["label"] for r in records])
    counts = np.array([r["count"] for r in records])
    # class-balance base weight
    n_pos, n_neg = (labels==1).sum(), (labels==0).sum()
    base = np.where(labels==1, n_neg/max(n_pos,1), 1.0).astype(np.float64)
    # up-weight positives by monument_count (capped)
    a = CONFIG["count_weight_alpha"]
    cw = np.where(labels==1, np.minimum(1 + a*np.log1p(counts),
                                        CONFIG["count_weight_cap"]), 1.0)
    return base * cw

print("TerrainDataset ready.")


TerrainDataset ready.


## 5 — Model: CNN encoder + gated attention (MIL) pooling head

In [5]:
# ============================================================
# CELL 5 — MODEL
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        red = max(ch // r, 4)
        self.fc1 = nn.Linear(ch, red); self.fc2 = nn.Linear(red, ch)
    def forward(self, x):
        b, c, _, _ = x.shape
        y = x.mean((2, 3))
        y = torch.sigmoid(self.fc2(F.relu(self.fc1(y)))).view(b, c, 1, 1)
        return x * y

class ConvBlock(nn.Module):
    def __init__(self, i, o, pool=True, se=True):
        super().__init__()
        self.c1 = nn.Conv2d(i, o, 3, padding=1, bias=False); self.b1 = nn.BatchNorm2d(o)
        self.c2 = nn.Conv2d(o, o, 3, padding=1, bias=False); self.b2 = nn.BatchNorm2d(o)
        self.se = SEBlock(o) if se else nn.Identity()
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()
    def forward(self, x):
        x = F.relu(self.b1(self.c1(x)))
        x = F.relu(self.b2(self.c2(x)))
        return self.pool(self.se(x))

class AttentionPool(nn.Module):
    # Gated-attention MIL pooling over the spatial feature map.
    # Aggregates HxW 'instances' with learned attention -> pooled vector.
    def __init__(self, ch, hidden=128):
        super().__init__()
        self.V = nn.Conv2d(ch, hidden, 1)
        self.U = nn.Conv2d(ch, hidden, 1)
        self.w = nn.Conv2d(hidden, 1, 1)
    def forward(self, x, valid=None):
        a = torch.tanh(self.V(x)) * torch.sigmoid(self.U(x))   # gated
        a = self.w(a)                                          # (B,1,H,W) logits
        if valid is not None:
            vm = F.interpolate(valid[:, None], size=x.shape[-2:], mode="nearest")
            # only mask patches that still have >=1 valid cell (avoid all -inf -> NaN)
            keep = vm.flatten(1).sum(1) > 0                    # (B,)
            mask = (vm < 0.5) & keep.view(-1, 1, 1, 1)
            a = a.masked_fill(mask, float("-inf"))
        b, _, h, wd = a.shape
        att = torch.softmax(a.view(b, -1), 1).view(b, 1, h, wd)
        pooled = (x * att).sum((2, 3))                         # (B,C)
        return pooled, att

class ArchNet(nn.Module):
    def __init__(self, in_ch=11, width=32, dropout=0.35, pool="attention"):
        super().__init__()
        self.enc = nn.Sequential(
            ConvBlock(in_ch, width, pool=True),
            ConvBlock(width, width*2, pool=True),
            ConvBlock(width*2, width*4, pool=False),
            ConvBlock(width*4, width*8, pool=False),
        )
        self.pool_kind = pool
        feat = width*8
        if pool == "attention":
            self.att = AttentionPool(feat)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(feat, 64), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x, valid=None, return_att=False):
        f = self.enc(x)
        if self.pool_kind == "attention":
            pooled, att = self.att(f, valid)
        else:
            pooled, att = f.mean((2, 3)), None
        out = self.head(self.drop(pooled)).squeeze(1)
        return (out, att) if return_att else out

# sanity check
_m = ArchNet(C, CONFIG["base_width"], CONFIG["dropout"], CONFIG["pool"]).to(DEVICE)
_x = torch.randn(2, C, P, P).to(DEVICE); _v = torch.ones(2, P, P).to(DEVICE)
print("forward:", _m(_x, _v).shape, "| params:",
      f"{sum(p.numel() for p in _m.parameters()):,}")
del _m, _x, _v


forward: torch.Size([2]) | params: 1,280,254


## 6 — Train / eval loops

In [6]:
# ============================================================
# CELL 6 — EPOCH RUNNER
# ============================================================
from sklearn.metrics import roc_auc_score, f1_score

def run_epoch(model, loader, criterion, optimizer=None, scaler=None, train=False):
    model.train() if train else model.eval()
    tot, probs, labs = 0.0, [], []
    for x, y, v in loader:
        x, y, v = x.to(DEVICE), y.to(DEVICE), v.to(DEVICE)
        if train: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type=="cuda")):
                logits = model(x, v)
                loss = criterion(logits, y)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                scaler.step(optimizer); scaler.update()
        tot += loss.item() * x.size(0)
        probs += torch.sigmoid(logits).detach().float().cpu().tolist()
        labs  += y.detach().cpu().tolist()
    probs, labs = np.array(probs), np.array(labs)
    auc = roc_auc_score(labs, probs) if len(set(labs.tolist()))==2 else float("nan")
    f1 = f1_score(labs, probs>=0.5, zero_division=0)
    return tot/len(loader.dataset), auc, f1, probs, labs
print("run_epoch ready.")


run_epoch ready.


## 7 — 5-fold group CV → OOF + ensemble on frozen test

In [7]:
# ============================================================
# CELL 7 — CROSS-VALIDATED TRAINING
# ============================================================
from sklearn.model_selection import StratifiedGroupKFold

X_idx  = np.arange(len(trainval))
y_all  = np.array([r["label"] for r in trainval])
g_all  = pd.factorize([str(r["group"]) for r in trainval])[0]

sgkf = StratifiedGroupKFold(n_splits=CONFIG["n_folds"], shuffle=True,
                            random_state=CONFIG["seed"])
oof_probs = np.zeros(len(trainval)); oof_lab = y_all.copy()
test_prob_folds = []

test_ds = TerrainDataset(test_recs, CH_MEAN, CH_STD, augment=False)
test_ld = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                     num_workers=2, pin_memory=True)
y_test = np.array([r["label"] for r in test_recs])

for fold, (tr, va) in enumerate(sgkf.split(X_idx, y_all, g_all), 1):
    tr_recs = [trainval[i] for i in tr]; va_recs = [trainval[i] for i in va]
    tr_ds = TerrainDataset(tr_recs, CH_MEAN, CH_STD, augment=True)
    va_ds = TerrainDataset(va_recs, CH_MEAN, CH_STD, augment=False)

    if CONFIG["use_weighted_sampler"]:
        w = make_sample_weights(tr_recs)
        sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                        num_samples=len(w), replacement=True)
        tr_ld = DataLoader(tr_ds, batch_size=CONFIG["batch_size"], sampler=sampler,
                           num_workers=2, pin_memory=True)
    else:
        tr_ld = DataLoader(tr_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                           num_workers=2, pin_memory=True)
    va_ld = DataLoader(va_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                       num_workers=2, pin_memory=True)

    model = ArchNet(C, CONFIG["base_width"], CONFIG["dropout"], CONFIG["pool"]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()   # imbalance handled by sampler
    optim_ = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                               weight_decay=CONFIG["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optim_, "max", factor=0.5,
                                                       patience=3, min_lr=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type=="cuda"))

    best_auc, best_state, no_imp = -1, None, 0
    for ep in range(1, CONFIG["epochs"]+1):
        print(ep)
        run_epoch(model, tr_ld, criterion, optim_, scaler, train=True)
        vl, vauc, vf1, _, _ = run_epoch(model, va_ld, criterion, train=False)
        sched.step(vauc)
        if vauc > best_auc + 1e-4:
            best_auc, best_state, no_imp = vauc, {k: v.detach().cpu().clone()
                                                  for k, v in model.state_dict().items()}, 0
        else:
            no_imp += 1
            if no_imp >= CONFIG["early_stop_patience"]: break
    model.load_state_dict(best_state)

    # OOF preds
    _, _, _, vp, _ = run_epoch(model, va_ld, criterion, train=False)
    oof_probs[va] = vp
    # test preds (this fold)
    _, _, _, tp, _ = run_epoch(model, test_ld, criterion, train=False)
    test_prob_folds.append(tp)
    torch.save(best_state, f"{CONFIG['ckpt_dir']}/fold{fold}.pt")
    print(f"Fold {fold}: best val AUC={best_auc:.4f}")

print("\nCV training done.")


1
2
3
4
5
6
7
8
9
Fold 1: best val AUC=0.9216
1
2
3
4
5
6
7
8
9
Fold 2: best val AUC=0.5917
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
Fold 3: best val AUC=0.6578
1
2
3
4
5
6
7
8
9
10
11
Fold 4: best val AUC=0.7313
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
Fold 5: best val AUC=0.6884

CV training done.


## 8 — Results: OOF AUC, threshold, ensemble test metrics

In [8]:
# ============================================================
# CELL 8 — FINAL EVALUATION
# ============================================================
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

oof_auc = roc_auc_score(oof_lab, oof_probs)
oof_ap  = average_precision_score(oof_lab, oof_probs)
# pick threshold maximizing OOF F1
best_t, best_f1 = 0.5, -1
for t in CONFIG["threshold_grid"]:
    f = f1_score(oof_lab, oof_probs>=t, zero_division=0)
    if f > best_f1: best_f1, best_t = f, t

ens = np.mean(test_prob_folds, axis=0)
pred = (ens >= best_t).astype(int)
print("="*60)
print("REFERENCE LINES:  LogReg 0.657 | best plain CNN ~0.68")
print("="*60)
print(f"OOF ROC-AUC   : {oof_auc:.4f}")
print(f"OOF PR-AUC    : {oof_ap:.4f}")
print(f"OOF F1 thresh : {best_t:.2f} (F1={best_f1:.3f})")
print("-"*60)
print("FROZEN TEST (5-fold ensemble):")
print(f"  ROC-AUC   : {roc_auc_score(y_test, ens):.4f}")
print(f"  PR-AUC    : {average_precision_score(y_test, ens):.4f}")
print(f"  Accuracy  : {accuracy_score(y_test, pred):.4f}")
print(f"  Precision : {precision_score(y_test, pred, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test, pred, zero_division=0):.4f}")
print(f"  F1        : {f1_score(y_test, pred, zero_division=0):.4f}")
print("  Confusion (rows=true [no_site,site]):")
print(confusion_matrix(y_test, pred))

# survey-prioritization view: recall in top-k% ranked patches
print("-"*60)
print("Recall @ top-k% highest-scored test patches (the deliverable metric):")
order = np.argsort(-ens)
for k in [0.10, 0.20, 0.30]:
    n = max(1, int(k*len(ens)))
    hit = y_test[order[:n]].sum()
    print(f"  top {int(k*100):2d}% ({n:3d} patches): "
          f"{int(hit)}/{int(y_test.sum())} sites  (recall={hit/y_test.sum():.3f})")


REFERENCE LINES:  LogReg 0.657 | best plain CNN ~0.68
OOF ROC-AUC   : 0.6612
OOF PR-AUC    : 0.4197
OOF F1 thresh : 0.52 (F1=0.516)
------------------------------------------------------------
FROZEN TEST (5-fold ensemble):
  ROC-AUC   : 0.5999
  PR-AUC    : 0.4092
  Accuracy  : 0.4180
  Precision : 0.3242
  Recall    : 0.8667
  F1        : 0.4719
  Confusion (rows=true [no_site,site]):
[[ 79 271]
 [ 20 130]]
------------------------------------------------------------
Recall @ top-k% highest-scored test patches (the deliverable metric):
  top 10% ( 50 patches): 24/150 sites  (recall=0.160)
  top 20% (100 patches): 42/150 sites  (recall=0.280)
  top 30% (150 patches): 56/150 sites  (recall=0.373)


## 9 — How to read this

Compare **OOF ROC-AUC** to the locked reference lines:

- **> ~0.72** — attention pooling + count-weighting found spatial signal GAP was washing
  out. Worth pushing further (Phase 2: SSL pretraining on tiled unlabeled patches).
- **~0.66–0.70** — you've matched/slightly beaten the linear + plain-CNN baselines by
  fixing the bugs, but the ceiling holds. Encoder changes won't add much; focus on labels.
- **≈ 0.66 and flat** — confirms the EDA: the ceiling is the labels/resolution, not the
  model. Pivot to the **top-k% recall** framing (survey prioritization) and/or cleaner
  negatives / higher-res inputs.

Change ONE thing at a time (pool="gap" vs "attention", count weighting on/off) and keep
this same group-CV harness so numbers stay comparable.


In [9]:
# ============================================================
# CELL A — SSL CONFIG + UNLABELED POOL
# ============================================================
import os, glob, numpy as np, torch, rasterio

SSL = {
    "mask_ratio": 0.60,       # fraction of patch tokens masked in MAE
    "patch_token": 25,        # tokenize 250x250 into 10x10 grid of 25px tokens
    "epochs": 30,
    "batch_size": 32,
    "lr": 1.5e-4,
    "weight_decay": 0.05,
    "ckpt": "/kaggle/working/ssl_encoder.pt",
}

# --- gather an UNLABELED pool -------------------------------------------------
# Prefer full source rasters if present; else fall back to the labeled patches.
raster_candidates = []
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith((".tif", ".tiff")) and "train" not in root.lower() \
           and "validation" not in root.lower() and "test" not in root.lower():
            raster_candidates.append(os.path.join(root, f))

def tile_raster(path, size=250, stride=250, max_tiles=4000):
    tiles = []
    with rasterio.open(path) as src:
        if src.count < CONFIG["n_channels"]:
            return tiles
        H, W = src.height, src.width
        for y in range(0, H - size + 1, stride):
            for x in range(0, W - size + 1, stride):
                tiles.append((path, x, y, size))
                if len(tiles) >= max_tiles:
                    return tiles
    return tiles

unlabeled_tiles = []
if raster_candidates:
    print(f"Found {len(raster_candidates)} candidate source rasters; tiling...")
    for rp in raster_candidates:
        unlabeled_tiles += tile_raster(rp)
    USE_SOURCE_RASTERS = len(unlabeled_tiles) > 0
else:
    USE_SOURCE_RASTERS = False

if not USE_SOURCE_RASTERS:
    print("No source rasters found — falling back to labeled patches as the SSL pool.")
    unlabeled_tiles = [(r["path"], None, None, None) for r in (trainval + test_recs)]

print(f"Unlabeled SSL pool: {len(unlabeled_tiles)} patches "
      f"({'source rasters' if USE_SOURCE_RASTERS else 'labeled patches'})")

No source rasters found — falling back to labeled patches as the SSL pool.
Unlabeled SSL pool: 3617 patches (labeled patches)


In [10]:
# ============================================================
# CELL B — SSL DATASET
# ============================================================
from torch.utils.data import Dataset, DataLoader

C, P = CONFIG["n_channels"], CONFIG["patch_size"]
NODATA = CONFIG["nodata_value"]
_mean = CH_MEAN.reshape(C, 1, 1)
_std  = CH_STD.reshape(C, 1, 1)

class SSLPatchDataset(Dataset):
    def __init__(self, tiles):
        self.tiles = tiles
    def __len__(self): return len(self.tiles)
    def _norm(self, arr):
        invalid = (~np.isfinite(arr)) | (arr == NODATA)
        arr = np.where(invalid, np.broadcast_to(_mean, arr.shape), arr).astype(np.float32)
        return ((arr - _mean) / _std).astype(np.float32)
    def __getitem__(self, i):
        path, x, y, size = self.tiles[i]
        with rasterio.open(path) as src:
            if x is None:
                arr = src.read().astype(np.float32)
            else:
                from rasterio.windows import Window
                arr = src.read(window=Window(x, y, size, size)).astype(np.float32)
        arr = arr[:C]
        if arr.shape[1:] != (P, P):
            t = np.full((C, P, P), np.nan, np.float32)
            h = min(P, arr.shape[1]); w = min(P, arr.shape[2])
            t[:, :h, :w] = arr[:, :h, :w]; arr = t
        return torch.from_numpy(self._norm(arr)).float()

ssl_ds = SSLPatchDataset(unlabeled_tiles)
ssl_ld = DataLoader(ssl_ds, batch_size=SSL["batch_size"], shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
print("SSL dataset ready:", len(ssl_ds), "patches")

SSL dataset ready: 3617 patches


In [11]:
# ============================================================
# CELL C — RESNET-50 ENCODER (11-ch) + MAE DECODER
# ============================================================
import torch.nn as nn, torch.nn.functional as F
import torchvision

def build_resnet50_encoder(in_ch=11, pretrained_rgb=True):
    """ResNet-50 adapted to 11 channels. If pretrained_rgb, inflate the ImageNet
    stem: copy the mean of the 3 RGB filters across all 11 input channels so the
    pretrained weights are a sane starting point instead of random."""
    weights = torchvision.models.ResNet50_Weights.DEFAULT if pretrained_rgb else None
    net = torchvision.models.resnet50(weights=weights)
    old = net.conv1  # (64,3,7,7)
    new = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)
    with torch.no_grad():
        if pretrained_rgb:
            w = old.weight.mean(dim=1, keepdim=True)          # (64,1,7,7)
            new.weight.copy_(w.repeat(1, in_ch, 1, 1) * (3.0 / in_ch))
    net.conv1 = new
    net.fc = nn.Identity()                                    # feature extractor -> 2048
    return net

class MAEWrapper(nn.Module):
    """Masked-autoencoding around the ResNet-50 feature map.
    Masks input tokens (25px blocks), encodes, and reconstructs the FULL patch
    from the 2048-d feature via a small conv decoder. Loss only on masked cells."""
    def __init__(self, encoder, in_ch=11, token=25):
        super().__init__()
        self.encoder = encoder
        self.token = token
        self.grid = P // token
        self.decoder = nn.Sequential(
            nn.Linear(2048, 512), nn.GELU(),
            nn.Linear(512, in_ch * token * token)
        )
    def random_mask(self, B, device):
        n = self.grid * self.grid
        keep = int(n * (1 - SSL["mask_ratio"]))
        idx = torch.rand(B, n, device=device).argsort(1)
        m = torch.ones(B, n, device=device); m.scatter_(1, idx[:, :keep], 0)
        return m  # 1 = masked
    def forward(self, x):
        B = x.size(0); dev = x.device
        m = self.random_mask(B, dev)                          # (B, grid*grid)
        mm = m.view(B, 1, self.grid, self.grid)
        mm = F.interpolate(mm, scale_factor=self.token, mode="nearest")  # (B,1,P,P)
        x_masked = x * (1 - mm)                                # zero the masked blocks
        feat = self.encoder(x_masked)                          # (B,2048)
        recon = self.decoder(feat).view(B, C, self.token, self.token)
        recon = recon.repeat_interleave(1, 0)                 # (B,C,token,token) per-block mean recon
        # upsample block-recon to full patch by tiling the predicted block pattern
        recon_full = F.interpolate(recon, size=(P, P), mode="nearest")
        loss = ((recon_full - x) ** 2)
        loss = (loss * mm).sum() / (mm.sum() * C + 1e-6)      # masked-only MSE
        return loss

print("ResNet-50 encoder + MAE wrapper defined.")

ResNet-50 encoder + MAE wrapper defined.


In [12]:
# ============================================================
# CELL D — SSL PRE-TRAIN THE RESNET-50 ENCODER
# ============================================================
ssl_encoder = build_resnet50_encoder(in_ch=C, pretrained_rgb=True).to(DEVICE)
mae = MAEWrapper(ssl_encoder, in_ch=C, token=SSL["patch_token"]).to(DEVICE)
opt = torch.optim.AdamW(mae.parameters(), lr=SSL["lr"], weight_decay=SSL["weight_decay"])
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

print(f"SSL pre-training ResNet-50 for {SSL['epochs']} epochs on {len(ssl_ds)} patches...")
for ep in range(1, SSL["epochs"] + 1):
    mae.train(); tot = 0.0
    for x in ssl_ld:
        x = x.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            loss = mae(x)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * x.size(0)
    if ep % 5 == 0 or ep == 1:
        print(f"  epoch {ep:2d}  recon MSE = {tot/len(ssl_ds):.4f}")

torch.save(ssl_encoder.state_dict(), SSL["ckpt"])
print("Saved SSL encoder ->", SSL["ckpt"])

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 164MB/s] 


SSL pre-training ResNet-50 for 30 epochs on 3617 patches...
  epoch  1  recon MSE = 0.9453
  epoch  5  recon MSE = 0.7628
  epoch 10  recon MSE = 0.6527
  epoch 15  recon MSE = 0.6189
  epoch 20  recon MSE = 0.5883
  epoch 25  recon MSE = 0.5714
  epoch 30  recon MSE = 0.5379
Saved SSL encoder -> /kaggle/working/ssl_encoder.pt


In [ ]:
# ============================================================
# CELL E — FINE-TUNE RESNET-50 IN GROUP-CV  (SSL vs from-scratch)
# ============================================================
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix)
import pandas as pd, numpy as np, torch.nn as nn

class ResNetClassifier(nn.Module):
    def __init__(self, ssl_ckpt=None):
        super().__init__()
        self.encoder = build_resnet50_encoder(in_ch=C, pretrained_rgb=(ssl_ckpt is None))
        if ssl_ckpt is not None:
            self.encoder.load_state_dict(torch.load(ssl_ckpt, map_location="cpu"))
            print("  loaded SSL weights")
        self.head = nn.Sequential(nn.Dropout(CONFIG["dropout"]), nn.Linear(2048, 1))
    def forward(self, x, valid=None):           # valid ignored (kept for harness compat)
        return self.head(self.encoder(x)).squeeze(1)

def run_resnet_cv(ssl_ckpt, tag):
    y_all = np.array([r["label"] for r in trainval])
    g_all = pd.factorize([str(r["group"]) for r in trainval])[0]
    sgkf = StratifiedGroupKFold(n_splits=CONFIG["n_folds"], shuffle=True,
                                random_state=CONFIG["seed"])
    oof = np.zeros(len(trainval)); test_folds = []
    y_test = np.array([r["label"] for r in test_recs])
    test_ld = DataLoader(TerrainDataset(test_recs, CH_MEAN, CH_STD, augment=False),
                         batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2)

    for fold, (tr, va) in enumerate(sgkf.split(np.arange(len(trainval)), y_all, g_all), 1):
        tr_recs = [trainval[i] for i in tr]; va_recs = [trainval[i] for i in va]
        w = make_sample_weights(tr_recs)
        from torch.utils.data import WeightedRandomSampler
        sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), len(w), True)
        tr_ld = DataLoader(TerrainDataset(tr_recs, CH_MEAN, CH_STD, augment=True),
                           batch_size=CONFIG["batch_size"], sampler=sampler, num_workers=2)
        va_ld = DataLoader(TerrainDataset(va_recs, CH_MEAN, CH_STD, augment=False),
                           batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2)

        model = ResNetClassifier(ssl_ckpt=ssl_ckpt).to(DEVICE)
        crit = nn.BCEWithLogitsLoss()
        opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                                weight_decay=CONFIG["weight_decay"])
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, "max", factor=0.5, patience=3)
        scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

        best, best_state, no_imp = -1, None, 0
        for ep in range(1, CONFIG["epochs"] + 1):
            print(ep)
            run_epoch(model, tr_ld, crit, opt, scaler, train=True)
            _, vauc, _, _, _ = run_epoch(model, va_ld, crit, train=False)
            sched.step(vauc)
            if vauc > best + 1e-4:
                best, best_state, no_imp = vauc, {k: v.cpu().clone()
                                                  for k, v in model.state_dict().items()}, 0
            else:
                no_imp += 1
                if no_imp >= CONFIG["early_stop_patience"]: break
        model.load_state_dict(best_state)
        _, _, _, vp, _ = run_epoch(model, va_ld, crit, train=False); oof[va] = vp
        _, _, _, tp, _ = run_epoch(model, test_ld, crit, train=False); test_folds.append(tp)
        print(f"  [{tag}] fold {fold}: val AUC={best:.4f}")

    oof_auc = roc_auc_score(y_all, oof)
    ens = np.mean(test_folds, 0)
    print(f"\n[{tag}] OOF ROC-AUC = {oof_auc:.4f} | "
          f"TEST ens AUC = {roc_auc_score(y_test, ens):.4f} | "
          f"PR-AUC = {average_precision_score(y_test, ens):.4f}")
    order = np.argsort(-ens)
    for k in [0.10, 0.20, 0.30]:
        n = max(1, int(k*len(ens))); hit = y_test[order[:n]].sum()
        print(f"    recall@top{int(k*100)}% = {hit/y_test.sum():.3f}")
    return oof_auc

print("REFERENCE:  LogReg 0.657 | plain CNN ~0.68 | V6 attn-CNN 0.651\n")
print("=== ResNet-50 FROM SCRATCH (ImageNet-inflated stem) ===")
auc_scratch = run_resnet_cv(ssl_ckpt=None, tag="scratch")
print("\n=== ResNet-50 WITH SSL PRE-TRAINING ===")
auc_ssl = run_resnet_cv(ssl_ckpt=SSL["ckpt"], tag="ssl")

print("\n" + "="*55)
print(f"SSL lift: {auc_ssl - auc_scratch:+.4f} OOF AUC "
      f"({auc_scratch:.4f} -> {auc_ssl:.4f})")
print("="*55)

REFERENCE:  LogReg 0.657 | plain CNN ~0.68 | V6 attn-CNN 0.651

=== ResNet-50 FROM SCRATCH (ImageNet-inflated stem) ===
1
2
3
4
5
6
7
8
9
10
  [scratch] fold 1: val AUC=0.9020
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
  [scratch] fold 2: val AUC=0.7097
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
  [scratch] fold 3: val AUC=0.7873
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
  [scratch] fold 4: val AUC=0.8251
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
  [scratch] fold 5: val AUC=0.7869

[scratch] OOF ROC-AUC = 0.7467 | TEST ens AUC = 0.7538 | PR-AUC = 0.6648
    recall@top10% = 0.287
    recall@top20% = 0.500
    recall@top30% = 0.560

=== ResNet-50 WITH SSL PRE-TRAINING ===
  loaded SSL weights
1
2
3
4
5
6
7
8
9
  [ssl] fold 1: val AUC=0.9608
  loaded SSL weights
1
2
3
4
5
6
7
8
9
10
11
12
  [ssl] fold 2: val AUC=0.6134
  loaded SSL weights
1
2
3
4
5
6
7
8
9
10
11
  [ssl] fold 3: val AUC=0.6602
  loaded SSL weights
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21

In [ ]:
# ============================================================
# CELL H — SAVE WHATEVER RESULTS EXIST (no retrain needed)
# ============================================================
import pickle, json, os, numpy as np, datetime

SAVE_DIR = "/kaggle/working"; os.makedirs(SAVE_DIR, exist_ok=True)

def clean(res):
    return {"tag": str(res["tag"]),
            "oof": np.asarray(res["oof"], np.float32),
            "oof_lab": np.asarray(res["oof_lab"], np.int64),
            "test_ens": np.asarray(res["test_ens"], np.float32),
            "test_lab": np.asarray(res["test_lab"], np.int64),
            "oof_auc": float(res["oof_auc"])}

bundle = {"meta": {
    "saved_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "reference_lines": {"logreg": 0.657, "cnn": 0.68, "v6_attn_cnn": 0.651},
}}

# full dicts if the E-patch was applied; else fall back to the bare AUC floats
if "res_scratch" in globals() and "res_ssl" in globals():
    bundle["imagenet"] = clean(res_scratch)
    bundle["ssl"]      = clean(res_ssl)
    print("Saved FULL result dicts (predictions + AUC).")
else:
    bundle["imagenet_auc"] = float(auc_scratch) if "auc_scratch" in globals() else None
    bundle["ssl_auc"]      = float(auc_ssl)     if "auc_ssl"     in globals() else None
    print("res_* dicts not found — saving only the AUC floats.")
    print("  (apply the Cell E patch and rerun E to capture predictions.)")

path = os.path.join(SAVE_DIR, "resnet_results.pkl")
with open(path, "wb") as f: pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(SAVE_DIR, "resnet_results_summary.json"), "w") as f:
    json.dump({k: (v if not isinstance(v, dict) or k=="meta"
                   else {"oof_auc": v.get("oof_auc")}) for k,v in bundle.items()},
              f, indent=2, default=str)

with open(path, "rb") as f: chk = pickle.load(f)
print(f"Saved & verified -> {path} ({os.path.getsize(path)/1024:.1f} KB)")
print("keys:", list(chk.keys()))

In [ ]:
# ============================================================
# CELL F(lite) — SUMMARY OF WHAT'S AVAILABLE (no retrain)
# ============================================================
# Your Cell E returned only OOF AUC floats (auc_scratch, auc_ssl).
# A full classification_report needs per-patch predictions, which were
# not captured. This prints the AUC-level summary you do have.
import pandas as pd

rows = []
if "auc_scratch" in globals():
    rows.append(("resnet_imagenet", auc_scratch))
if "auc_ssl" in globals():
    rows.append(("resnet_ssl", auc_ssl))

df = pd.DataFrame(rows, columns=["model", "OOF_ROC_AUC"])
ref = pd.DataFrame([("logreg_baseline", 0.657), ("plain_cnn", 0.68),
                    ("v6_attn_cnn", 0.651)], columns=["model", "OOF_ROC_AUC"])
out = pd.concat([ref, df], ignore_index=True).sort_values("OOF_ROC_AUC", ascending=False)
print(out.to_string(index=False))
print("\nNote: per-class precision/recall/F1 report is unavailable — Cell E did not")
print("keep predictions. Apply the 3-line Option-1 edit and rerun E to get the full report.")

In [ ]:
# ============================================================
# TEST-SET PRECISION / RECALL (exact, per class)
# ============================================================
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

def best_threshold(oof, lab, grid=np.arange(0.10, 0.71, 0.01)):
    best_t, best = 0.5, -1
    for t in grid:
        f = f1_score(lab, oof >= t, zero_division=0)
        if f > best: best, best_t = f, t
    return best_t

for res in [res_scratch, res_ssl]:
    lab = res["test_lab"]
    ens = res["test_ens"]
    t = best_threshold(res["oof"], res["oof_lab"])   # threshold picked on OOF, not test
    pred = (ens >= t).astype(int)

    print("=" * 50)
    print(f"{res['tag'].upper()}   (threshold = {t:.2f}, chosen on OOF)")
    print("-" * 50)
    # class 1 = site, class 0 = no_site
    for cls, name in [(1, "site"), (0, "no_site")]:
        p = precision_score(lab, pred, pos_label=cls, zero_division=0)
        r = recall_score(lab, pred, pos_label=cls, zero_division=0)
        print(f"  {name:8s}  precision={p:.3f}   recall={r:.3f}")
    tn, fp, fn, tp = confusion_matrix(lab, pred).ravel()
    print(f"  confusion: TP={tp} FP={fp} FN={fn} TN={tn}")
    print(f"  (site: caught {tp}/{tp+fn} real sites, "
          f"{fp} false alarms out of {fp+tn} no_site)")